# Week 5 — EDA I: distributions, missingness, outliers

### Deliverable
- 6–8 plots + 5 insights (each with a 'could be wrong because...')


In [ ]:
# Core imports (kept minimal for beginners)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Dataset URL (UCI Heart Disease - Cleveland)
UCI_URL = "https://www.archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Column names for processed.cleveland.data (14 columns commonly used in teaching)
COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","num"]


In [ ]:
import ssl
import io
import urllib.request # Added this import

def load_raw():
    # Create an unverified SSL context to bypass certificate verification
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    # Open the URL with the unverified context
    with urllib.request.urlopen(UCI_URL, context=ctx) as url_response:
        # Read the content and decode it
        s = url_response.read().decode('utf-8')

    # Use io.StringIO to make the string behave like a file for pandas.read_csv
    df_raw = pd.read_csv(io.StringIO(s), header=None, names=COLS)
    return df_raw

def coerce_types(df_raw):
    # Missing values are sometimes encoded as "?"
    df = df_raw.replace("?", np.nan).copy()

    # Convert numeric-looking columns
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Binary target: disease present if num > 0 (UCI convention)
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df

df_raw = load_raw()
df = coerce_types(df_raw)

df.head()


In [ ]:
def clean_heart(df_raw):
    df = df_raw.replace("?", np.nan).copy()
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df

df_clean = clean_heart(df_raw)


## Missingness profile


In [ ]:
miss = df_clean.isna().mean().sort_values(ascending=False)
display(miss.to_frame("missing_fraction"))

plt.figure()
miss.head(12).sort_values().plot(kind="barh")
plt.xlabel("missing fraction")
plt.title("Top missing columns")
plt.show()


## Numeric distributions (choose 4–5)


In [ ]:
numeric = ["age","trestbps","chol","thalach","oldpeak"]
for c in numeric:
    plt.figure()
    df_clean[c].dropna().plot(kind="hist", bins=20)
    plt.title(f"Distribution: {c}")
    plt.xlabel(c)
    plt.ylabel("count")
    plt.show()


## Outlier investigation (IQR rule as a starting point)


In [ ]:
col = "chol"  # TODO
x = df_clean[col].dropna()
q1, q3 = x.quantile(0.25), x.quantile(0.75)
iqr = q3 - q1
lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr

print("IQR bounds:", lo, hi)
outliers = df_clean[df_clean[col].between(lo, hi) == False][[col,"age","sex","disease"]].head(15)
outliers


## TODO — Write 5 insights
1. Age is roughly bell-shaped, centered around 54–55 years - this is not a general population sample. Most patients fall between 40 and 70, with very few under 35 or over 75. The dataset represents middle-aged to older adults referred for cardiac workup, not a random cross-section of society.

2. Cholesterol has extreme high-end outliers (some values above 400–560 mg/dL). The IQR rule flags several patients with very high cholesterol. These could be genuine cases of familial hypercholesterolemia, or they could be data entry errors (e.g., units recorded differently). They are worth inspecting before including in any model.

3. Oldpeak (ST depression) is strongly right-skewed - the majority of patients have zero or near-zero values. Most patients show no ST depression at rest, but a small subset shows very high values (>4 mm), suggesting severe ischemia. This skew means the mean is not a good summary statistic - the median would be more representative.

4. Thalach (maximum heart rate achieved) is left-skewed - few patients reach very low heart rates. The bulk of patients achieve 140–170 bpm, but a tail of low values exists. Lower max heart rate is generally associated with older age and worse cardiac function, consistent with the disease group.

5. Missingness is almost entirely confined to ca and thal (about 2% each)- all other columns are complete. This is a very clean dataset by real-world standards. The low and localized missingness suggests the data was carefully curated for research use. The pattern (only two columns missing, small count) is consistent with random equipment/recording failure rather than systematic omission.
